#### IDF + Word2Vec
- Word2Vec은 단문에서 효과적인 벡터화
- TF-IDF에서 TF는 하나의 문장에서 출현 횟수의 값
    - 단문에서 단어들의 출현 횟수는 일반적으로 1회 정도 → 큰 의미를 가질 수 없다.
- IDF와 Word2Vec을 혼합하여 활용

In [1]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from konlpy.tag import Komoran

In [2]:
docs = [
    '오늘 날씨가 좋다 여행 가고 싶다',
    '기온이 너무 올라서 아무것도 하기 싫다',
    '수업이 너무 지루하고 졸리다',
    '음식이 너무 맛이 없고 서비스도 별로다',
    '영화가 너무 재미있어서 시간이 가는 줄 몰랐다'
]

target = [1, 0, 0, 0, 1]

In [3]:
def tokenize(text):
    # konlpy 설치하고 토큰화 객체 생성 시 JDK 필요 (최신 버전에서 문제 발생)
    # Komoran이 사용 가능한 경우와 불가능한 경우
    try:
        komoran = Komoran()
        allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
        tokens = []
        for word, pos in komoran.pos(text):
            if pos in allow_pos:
                tokens.append(word)
    except Exception as e:
        print('Komoran 사용 불가:', e)
        tokens = text.split()
    
    return tokens

In [4]:
tokenize(docs[0])

['오늘', '날씨', '좋', '여행', '가']

In [5]:
tokens = []
for doc in docs:
    token = tokenize(doc)
    tokens.append(token)

tokens

[['오늘', '날씨', '좋', '여행', '가'],
 ['기온', '너무', '오르', '아무것', '하', '싫'],
 ['수업', '너무', '졸리'],
 ['음식', '너무', '맛', '없', '서비스', '별로', '다'],
 ['영화', '너무', '재미있', '시간', '가', '모르']]

In [6]:
# Word2Vec 생성

w2v = Word2Vec(
    sentences = tokens,
    vector_size = 100,
    window = 5,
    min_count = 1,
    epochs = 100,
    sg = 1,
    workers = 2,
    seed = 42
)

In [7]:
wv = w2v.wv

In [9]:
wv['오늘']

array([-2.7287216e-03, -9.8401727e-03, -9.1918800e-03, -3.4337200e-03,
       -9.6288335e-04,  6.0229236e-03, -7.4626431e-03,  6.8369680e-03,
       -5.4082973e-03, -7.3324060e-03,  3.9405078e-03,  5.6329258e-03,
        2.1988966e-03, -7.5499737e-03, -7.6068700e-03, -2.1043839e-03,
       -9.8573719e-04,  9.8119881e-03, -4.3554106e-03, -6.2052952e-03,
        8.8562779e-03, -6.8946718e-03, -9.5161535e-03, -1.4912968e-03,
       -1.7910032e-03,  2.3984711e-03,  7.4954885e-03,  8.7831849e-03,
       -4.2997450e-03,  9.6779829e-03,  2.6489382e-03, -8.6479262e-03,
        8.7077711e-03, -7.2242222e-03,  1.8151662e-03,  5.7141460e-03,
       -4.9147238e-03, -4.7990177e-03, -5.2592164e-04,  7.8458870e-03,
       -5.4515968e-03,  3.3419395e-03, -7.3593151e-04, -7.8239627e-03,
        4.1161766e-03,  6.6746152e-03,  1.4695618e-03, -6.3069672e-03,
       -6.6460473e-03, -1.5646811e-03,  1.4033304e-04,  8.6812972e-04,
       -5.2330559e-03,  8.8888126e-05, -4.1301786e-03,  3.6540648e-03,
      

In [10]:
wv.index_to_key

['너무',
 '가',
 '모르',
 '시간',
 '재미있',
 '영화',
 '다',
 '별로',
 '서비스',
 '없',
 '맛',
 '음식',
 '졸리',
 '수업',
 '싫',
 '하',
 '아무것',
 '오르',
 '기온',
 '여행',
 '좋',
 '날씨',
 '오늘']

In [11]:
# 문장을 입력값으로 단위 벡터의 평균을 구하는 함수
def sent_embed_mean(token):
    # token: 토큰화된 하나의 문장
    vector = []
    for word in token:
        if word in wv.index_to_key:
        # Word2Vec에서 학습된 단어 사전에 word가 존재한다면
        # vector 리스트에 해당 단어의 단위 벡터를 추가
            vector.append(wv[word])
        
    # 만약 새로운 문장의 단어들이 Word2Vec에서 사전에 학습된 단어 사전에 존재하지 않는 경우
        # vector 값이 빈 리스트
    if vector:
        result = np.mean(vector, axis = 0)
    else:
        # vector가 존재하지 않는 경우에는 영행렬을 생성
        result = np.zeros(wv.vector_size)
    
    return result

In [12]:
# tokens 데이터를 이용하여 단위 벡터 평균 함수 호출
X_embed_wv = []
for token in tokens:
    X_embed_wv.append(
        sent_embed_mean(token)
    )

In [16]:
X_embed_wv

[array([ 4.0627131e-03, -4.1177319e-03, -1.4225660e-03, -5.4318546e-03,
         2.9331266e-03, -1.0025300e-03, -5.0470594e-04, -2.1184415e-03,
        -2.8752084e-03,  6.8741088e-04, -2.7533637e-03,  4.1878768e-03,
        -9.9428801e-04,  2.8869177e-03, -5.6480215e-04, -1.9054037e-03,
        -1.4098274e-03,  1.4024605e-03, -1.1003042e-03, -2.0396772e-03,
         1.3082147e-03,  1.9095670e-03,  9.8853686e-04,  2.0640534e-04,
        -7.6428616e-05, -1.8237742e-03, -1.0836786e-03,  3.7147789e-03,
        -3.5496112e-03,  2.9948461e-03,  2.3482223e-04, -1.0175806e-03,
        -1.9324823e-04,  9.5323147e-04,  1.8641306e-03,  1.0443751e-03,
        -4.8639177e-04, -6.5589370e-03,  3.1723916e-03, -3.4027320e-04,
        -4.4211504e-04,  1.6320575e-03,  3.4491390e-03, -2.4417550e-03,
         8.0322695e-04,  2.9438485e-03, -4.4540949e-03,  1.3637433e-03,
         5.6419882e-04, -8.0012623e-04,  1.1474579e-03, -1.9369501e-03,
        -3.2696090e-04,  1.0688237e-03,  2.4639149e-03,  3.46896

In [17]:
# 데이터 train, test 분할 / 생성된 모델을 매개변수로 받아서 학습, 예측 후 평가 지표 출력

def run_model(X, y, model, test_size = 0.2, stratify = None):
    # X: 독립 변수
    # y: 종속 변수
    # model: 사용할 모델
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = test_size, stratify = stratify, random_state = 42
    )

    # 인자로 받은 모델을 이용해서 학습
    model.fit(X_train, y_train)
    # 학습된 모델을 이용하여 예측값을 생성
    pred = model.predict(X_test)

    result = classification_report(pred, y_test)
    print(result)

In [18]:
svc = SVC(random_state = 42)

In [19]:
run_model(X_embed_wv, target, svc)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       1.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0



c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [18]:
# Word2Vec에 idf의 값들을 포함
# 문맥 상에서 단어의 예측 벡터와 전체 문서에서 특정 단어들의 중요도를 결합

# TF-IDF 벡터화 행렬 생성
tfidf_vec = TfidfVectorizer(
    tokenizer = tokenize,
    lowercase = False
).fit(docs)

c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [19]:
# word2vec에서는 단어의 이름을 가지고 단위 벡터 생성
# idf 값들도 단어에 따라서 추출하기 편하게 만들기 위해서 dict 형태로 데이터 생성

idf_weight = dict(zip(
    tfidf_vec.get_feature_names_out(),
    tfidf_vec.idf_
))

idf_weight

{'가': np.float64(1.6931471805599454),
 '기온': np.float64(2.09861228866811),
 '날씨': np.float64(2.09861228866811),
 '너무': np.float64(1.1823215567939547),
 '다': np.float64(2.09861228866811),
 '맛': np.float64(2.09861228866811),
 '모르': np.float64(2.09861228866811),
 '별로': np.float64(2.09861228866811),
 '서비스': np.float64(2.09861228866811),
 '수업': np.float64(2.09861228866811),
 '시간': np.float64(2.09861228866811),
 '싫': np.float64(2.09861228866811),
 '아무것': np.float64(2.09861228866811),
 '없': np.float64(2.09861228866811),
 '여행': np.float64(2.09861228866811),
 '영화': np.float64(2.09861228866811),
 '오늘': np.float64(2.09861228866811),
 '오르': np.float64(2.09861228866811),
 '음식': np.float64(2.09861228866811),
 '재미있': np.float64(2.09861228866811),
 '졸리': np.float64(2.09861228866811),
 '좋': np.float64(2.09861228866811),
 '하': np.float64(2.09861228866811)}

In [20]:
# 단어별 단위 벡터의 평균 값과 idf 역수의 값들을 곱하여 새로운 벡터를 생성
def sent_embed_wv_idf(token):
    # 단어별 단위벡터
    vector = []
    # idf 값
    idf = []

    for word in token:
        if word in wv.key_to_index and word in idf_weight:
            vector.append(wv[word]*idf_weight[word])
            idf.append(idf_weight[word])
            # 각 단어별 단위 벡터에 idf를 곱한 값 → vector의 합산과 idf의 합산을 나눠준다. (평균을 구하는 방식)

    if vector:
        result = np.sum(vector, axis = 0) / (np.sum(idf) + 1e-9)
    else:
        result = np.zeros(wv.vector_size)
    
    return result

In [21]:
X_emb_wv_idf = []
for token in tokens:
    X_emb_wv_idf.append(
        sent_embed_wv_idf(token)
    )

X_emb_wv_idf

[array([ 0.00396025, -0.0040432 , -0.00172925, -0.00524193,  0.0028172 ,
        -0.00127677, -0.00074123, -0.00234184, -0.0029608 ,  0.00055696,
        -0.00268157,  0.00412315, -0.00108426,  0.00302904, -0.00059519,
        -0.00204202, -0.00109562,  0.00175941, -0.00093413, -0.00180888,
         0.00141426,  0.00185019,  0.00090438,  0.00023593, -0.00037533,
        -0.00195164, -0.00078752,  0.00364375, -0.0037474 ,  0.0030018 ,
         0.00019191, -0.00110021,  0.00013349,  0.0009411 ,  0.00170003,
         0.0012532 , -0.00059357, -0.00643534,  0.0034267 , -0.00030963,
        -0.0008551 ,  0.00192751,  0.00375811, -0.00246221,  0.00044319,
         0.00277744, -0.0042555 ,  0.00162175,  0.00031505, -0.00047431,
         0.00090436, -0.00182786, -0.0006682 ,  0.0012758 ,  0.00261748,
         0.00023572,  0.00090302,  0.0026021 , -0.00046791,  0.00066545,
         0.00423717,  0.00393203, -0.00580099,  0.00387544,  0.00183005,
         0.00218161, -0.00010088,  0.00287496,  0.0

In [22]:
run_model(X_emb_wv_idf, target, svc)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       1.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0



c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [23]:
# 학습된 모델에서 예측 값들을 되돌려 받는 함수 생성
# embedding 기법 선택: w2v 만을 이용한 평균, w2v+idf 혼합한 벡터 평균

# 매개변수 3개 : 새로운 데이터(문장들), 학습된 모델, 벡터화 타입
def predict_sentence_list(
        sentences, model, vec_type = 'wv'
):
    # sentences: 문장들의 리스트
    # model: 학습된 모델
    # vec_type: 'wv' 또는 'idf' 값을 받는다.
    # 문장들을 토큰화 → tokenize 함수 호출하여 결과를 받아온다.

    X_test = []

    for sentence in sentences:
        token = tokenize(sentence)
        # 하나의 문장이 토큰화가 진행 되었으면 벡터화 함수에 데이터를 대입
        if vec_type == 'wv':
            vec = sent_embed_mean(token)
        elif vec_type == 'idf':
            vec = sent_embed_wv_idf(token)
        else:
            print('vec_type이 맞지 않습니다')
            return ''
        X_test.append(vec)  # 독립변수 생성 완료
    
    preds = model.predict(X_test)
    result = []
    
    for sentence, pred in zip(sentences, preds):
        label = '긍정' if pred == 1 else '부정'
        result.append([sentence, label])
    return result

In [24]:
new_sentences = [
    '영화가 너무 지루해서 돈이 아깝다',
    '날씨가 너무 별로다',
    '기온이 좋아서 어디론가 떠나고 싶다'
]

In [ ]:
predict_sentence_list(new_sentences, svc, 'wv')

[['영화가 너무 지루해서 돈이 아깝다', '긍정'],
 ['날씨가 너무 별로다', '부정'],
 ['기온이 좋아서 어디론가 떠나고 싶다', '긍정']]

In [26]:
predict_sentence_list(new_sentences, svc, 'idf')

[['영화가 너무 지루해서 돈이 아깝다', '긍정'],
 ['날씨가 너무 별로다', '부정'],
 ['기온이 좋아서 어디론가 떠나고 싶다', '긍정']]

In [35]:
a = 'Hello'

In [28]:
def func_1(text):
    a = text
    return a

In [29]:
print(a)
print(func_1('Hi'))

Hello
Hi


In [30]:
print(a)

Hello


In [31]:
def func_2(text):
    globals()['a'] = text
    return a

In [32]:
print(a)
print(func_1('Hi1'))
print(func_2('Hi2'))

Hello
Hi1
Hi2


In [33]:
print(a)

Hi2


In [34]:
def func_3(text):
    # a라는 전역 변수를 가지고 와서 사용
    global a
    a = text
    return a

In [36]:
print(a)
print(func_3('Hi'))

Hello
Hi


In [37]:
print(a)

Hi
